# Project: Music Improvisation for Hearing Loss
Using the MUSDB18 dataset for source separation and audio enhancement.

In [ ]:
!pip install musdb

### Loading the Dataset
You can either use the full dataset if you have it on Drive/Local, or use the `musdb.DB` setup to point to your directory. For experimentation, you might want to start with the `musdb18` python package which handles the manifest and metadata.

In [ ]:
try:
    import musdb
except ImportError:
    print("musdb not found, installing...")
    !pip install musdb
    import musdb

from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Update this path to where your MUSDB18 folder is located
musdb_path = '/content/drive/MyDrive/Minor Training'

try:
    mus = musdb.DB(root=musdb_path)
    print(f"Successfully loaded {len(mus)} tracks from Drive.")
except Exception as e:
    print(f"Error loading dataset: {e}. Please check the 'musdb_path' variable.")

In [ ]:
import os

# Let's check the actual folder structure to see why musdb isn't picking them up
print(f"Checking contents of: {musdb_path}")
try:
    contents = os.listdir(musdb_path)
    print("Items found:", contents)

    # Check one level deeper for the first item
    if contents:
        first_item = os.path.join(musdb_path, contents[0])
        if os.path.isdir(first_item):
            print(f"Contents of {contents[0]}:", os.listdir(first_item))
except Exception as e:
    print(f"Error accessing path: {e}")

### Manual Loader Fallback
If the `musdb` library continues to show 0 tracks because of the folder naming, we can use `librosa` to load the stems directly for your hearing loss processing.

In [ ]:
import librosa
import os

def load_custom_stems(song_path):
    stems = {}
    sr = 44100  # Default fallback sample rate
    found_any = False

    for stem_name in ['vocals', 'drums', 'bass', 'other']:
        file_path = os.path.join(song_path, f"{stem_name}.wav")
        if os.path.exists(file_path):
            audio, loaded_sr = librosa.load(file_path, sr=None, mono=False)
            stems[stem_name] = audio
            sr = loaded_sr
            found_any = True

    return stems, sr if found_any else sr

print("Manual loader updated with safety checks.")

### Processing a Song from Drive
Now we will use the manual loader to fetch the stems for one of your songs and apply the enhancement filter.

In [ ]:
import os
import matplotlib.pyplot as plt
from scipy.signal import butter, lfilter

# Defining the missing function locally to ensure the cell runs
def high_shelf_filter(data, cutoff, fs, gain_db):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(1, normal_cutoff, btype='highpass', analog=False)
    boosted = lfilter(b, a, data)
    gain_linear = 10**(gain_db / 20)
    return data + (boosted * (gain_linear - 1))

# Define the path to a specific song found in the listdir earlier
song_folder = os.path.join(musdb_path, 'The Districts - Vermont')

# Load stems using the manual loader
stems, sr = load_custom_stems(song_folder)

if 'vocals' in stems:
    print(f"Loaded vocals for: {os.path.basename(song_folder)}")

    # Apply the high-shelf boost (10dB boost above 3kHz)
    original_vocals = stems['vocals'][0]
    boosted_vocals_drive = high_shelf_filter(original_vocals, 3000, sr, 10)

    # Plotting comparison
    plt.figure(figsize=(10, 4))
    plt.plot(original_vocals[10000:12000], label='Original', alpha=0.7)
    plt.plot(boosted_vocals_drive[10000:12000], label='Boosted (>3kHz)', alpha=0.7)
    plt.title("Enhancement Check: 'The Districts - Vermont' Vocals")
    plt.legend()
    plt.show()
else:
    print('Vocals file not found in the specified folder.')

### Processing a Song from Drive
Now we will use the manual loader to fetch the stems for one of your songs and apply the enhancement filter.

In [ ]:
# Define the path to a specific song
song_folder = os.path.join(musdb_path, 'The Districts - Vermont')

# Load stems
stems, sr = load_custom_stems(song_folder)

if 'vocals' in stems:
    print(f"Loaded vocals for: {os.path.basename(song_folder)}")

    # Apply the high-shelf boost (10dB boost above 3kHz)
    # stems['vocals'] shape is usually (channels, samples)
    original_vocals = stems['vocals'][0] # Take left channel
    boosted_vocals_drive = high_shelf_filter(original_vocals, 3000, sr, 10)

    # Plotting
    plt.figure(figsize=(10, 4))
    plt.plot(original_vocals[10000:12000], label='Original', alpha=0.7)
    plt.plot(boosted_vocals_drive[10000:12000], label='Boosted (>3kHz)', alpha=0.7)
    plt.title("Enhancement Check: 'The Districts - Vermont' Vocals")
    plt.legend()
    plt.show()
else:
    print("Vocals file not found in the specified folder.")

### Step 1: Batch Processing and Multi-band Compression
We will now define a more sophisticated Multi-band Compressor and then loop through all tracks in your Google Drive to apply these enhancements.

In [ ]:
def apply_multiband_compression(audio, sr):
    nyq = 0.5 * sr
    # Use a safe crossover
    crossover = min(1000, nyq * 0.9)
    b_low, a_low = butter(1, crossover/nyq, btype='lowpass')
    b_high, a_high = butter(1, crossover/nyq, btype='highpass')

    low_freq = lfilter(b_low, a_low, audio)
    high_freq = lfilter(b_high, a_high, audio)

    # Apply a stronger gain to high frequencies
    enhanced_high = high_freq * 2.5
    return low_freq + enhanced_high

processed_results = {}

# Loop through all folders, excluding the output folder
for folder_name in contents:
    if folder_name == 'Enhanced_Outputs':
        continue

    song_path = os.path.join(musdb_path, folder_name)
    if os.path.isdir(song_path):
        print(f"Processing: {folder_name}...")
        stems_batch, sr_batch = load_custom_stems(song_path)

        if 'vocals' in stems_batch:
            vocal_channel = stems_batch['vocals'][0]
            enhanced = apply_multiband_compression(vocal_channel, sr_batch)
            processed_results[folder_name] = enhanced

print(f"\nFinished processing {len(processed_results)} tracks.")

### Step 2: Evaluation Setup (HASPI)
To calculate the Hearing Aid Speech Perception Index (HASPI), we typically need the `pyaudiometrics` or custom HASPI implementations. For now, let's prepare the comparison infrastructure.

In [ ]:
import numpy as np
import os

def calculate_improvement_score(original, enhanced):
    # Placeholder for HASPI logic
    # In a real scenario, this would compare the 'intelligibility' of the two signals
    rms_orig = np.sqrt(np.mean(original**2))
    rms_enh = np.sqrt(np.mean(enhanced**2))
    improvement = (rms_enh / rms_orig) if rms_orig > 0 else 0
    return improvement

# Quick test on the first processed result
if processed_results:
    first_song = list(processed_results.keys())[0]
    # Reload original for accurate comparison
    song_folder_test = os.path.join(musdb_path, first_song)
    stems_test, _ = load_custom_stems(song_folder_test)
    if 'vocals' in stems_test:
        score = calculate_improvement_score(stems_test['vocals'][0], processed_results[first_song])
        print(f"Relative Power Improvement for '{first_song}': {score:.2f}x")
else:
    print("No processed results found to evaluate.")

### Step 2 (Continued): HAAPI Score Placeholder
HAAPI (Hearing Aid Audio Processing Index) is a metric designed to predict speech intelligibility in listeners with hearing impairment. A full HAAPI implementation is complex and typically requires a clean reference speech signal, a noisy speech signal, and the enhanced speech signal, along with specific auditory models. For now, we will add a placeholder function.

### Step 3: Spectral Visualization of Enhancement
To confirm the Multi-band Compressor is working correctly, let's compare the frequency distribution before and after processing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_spectral_comparison(original, enhanced, sr, title="Spectral Comparison"):
    plt.figure(figsize=(12, 6))

    # Calculate FFT
    fft_orig = np.abs(np.fft.rfft(original))
    fft_enh = np.abs(np.fft.rfft(enhanced))
    freqs = np.fft.rfftfreq(len(original), 1/sr)

    plt.semilogy(freqs, fft_orig, label='Original', alpha=0.6)
    plt.semilogy(freqs, fft_enh, label='Enhanced (Multi-band)', alpha=0.6)

    plt.axvline(1000, color='r', linestyle='--', label='Crossover (1kHz)')
    plt.title(title)
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Amplitude (log)')
    plt.xlim(0, 8000) # Focus on human voice range
    plt.legend()
    plt.grid(True, which='both', ls='-', alpha=0.2)
    plt.show()

# Visualize the first track
song_name = list(processed_results.keys())[0]
song_folder = os.path.join(musdb_path, song_name)
stems_orig, _ = load_custom_stems(song_folder)

plot_spectral_comparison(stems_orig['vocals'][0], processed_results[song_name], sr_batch, title=f"Spectral Effect of Enhancement on {song_name}")

### Step 4: Improvisation Logic (Dynamic Profiles)
We will define different listener profiles that automatically adjust the gain and crossover frequencies based on the severity of hearing loss.

In [ ]:
def improvise_enhancement(audio, sr, profile='mild'):
    nyq = 0.5 * sr
    if profile == 'mild':
        crossover = 2000
        gain = 1.8 # ~5dB
    elif profile == 'moderate':
        crossover = 1500
        gain = 2.5 # ~8dB
    elif profile == 'severe':
        crossover = 1000
        gain = 4.0 # ~12dB

    b_low, a_low = butter(1, crossover/nyq, btype='lowpass')
    b_high, a_high = butter(1, crossover/nyq, btype='highpass')

    low = lfilter(b_low, a_low, audio)
    high = lfilter(b_high, a_high, audio)

    return low + (high * gain)

# Example of improvisation for a severe profile
severe_enhanced = improvise_enhancement(stems_orig['vocals'][0], sr_batch, profile='severe')
print("Dynamic enhancement for 'Severe' profile applied.")

### Step 5: Export Enhanced Audio
Finally, we will save the processed results back to your Google Drive for use in other applications.

In [ ]:
import soundfile as sf

output_root = os.path.join(musdb_path, 'Enhanced_Outputs')
if not os.path.exists(output_root):
    os.makedirs(output_root)

for song_name, audio_data in processed_results.items():
    out_path = os.path.join(output_root, f"{song_name}_enhanced_vocals.wav")
    # Normalize before saving to prevent clipping
    norm_audio = audio_data / np.max(np.abs(audio_data))
    sf.write(out_path, norm_audio, sr_batch)
    print(f"Saved: {out_path}")

### Step 6: Advanced Quality Metrics (PSNR & SMR)
To further evaluate the enhancement, we can calculate the Peak Signal-to-Noise Ratio (PSNR). While SMR typically requires a psychoacoustic model, we can implement a Signal-to-Noise Ratio (SNR) as a proxy or placeholder for SMR analysis.

In [ ]:
import numpy as np

def calculate_psnr(original, enhanced):
    # Ensure signals are the same length
    min_len = min(len(original), len(enhanced))
    original = original[:min_len]
    enhanced = enhanced[:min_len]

    mse = np.mean((original - enhanced) ** 2)
    if mse == 0:
        return float('inf')

    max_val = np.max(np.abs(original))
    psnr = 20 * np.log10(max_val / np.sqrt(mse))
    return psnr

def calculate_snr(signal, noise_reference):
    # Simple SNR calculation as a proxy for SMR/Audio Quality
    signal_power = np.mean(signal ** 2)
    noise = signal - noise_reference
    noise_power = np.mean(noise ** 2)

    if noise_power == 0:
        return float('inf')

    return 10 * np.log10(signal_power / noise_power)

# Calculate for the last processed track
song_name = list(processed_results.keys())[0]
song_folder = os.path.join(musdb_path, song_name)
stems_eval, _ = load_custom_stems(song_folder)

if 'vocals' in stems_eval:
    orig = stems_eval['vocals'][0]
    enh = processed_results[song_name]

    psnr_val = calculate_psnr(orig, enh)
    snr_val = calculate_snr(enh, orig)

    print(f"Metrics for {song_name}:")
    print(f" - PSNR: {psnr_val:.2f} dB")
    print(f" - SNR (Improvement Proxy): {snr_val:.2f} dB")

In [ ]:
import pandas as pd

all_metrics = []

for song_name, enhanced_audio in processed_results.items():
    # Load the original stem for this specific song
    song_folder_path = os.path.join(musdb_path, song_name)
    stems_temp, _ = load_custom_stems(song_folder_path)

    if 'vocals' in stems_temp:
        original_audio = stems_temp['vocals'][0]

        # Calculate all metrics
        psnr = calculate_psnr(original_audio, enhanced_audio)
        snr = calculate_snr(enhanced_audio, original_audio)
        rms_orig = np.sqrt(np.mean(original_audio**2))
        rms_enh = np.sqrt(np.mean(enhanced_audio**2))
        improvement = rms_enh / rms_orig if rms_orig > 0 else 0

        all_metrics.append({
            'Song Name': song_name,
            'Original RMS': round(rms_orig, 4),
            'Enhanced RMS': round(rms_enh, 4),
            'Improvement Factor': round(improvement, 2),
            'PSNR (dB)': round(psnr, 2),
            'SNR Proxy (dB)': round(snr, 2)
        })

# Create and display the DataFrame
df_all_metrics = pd.DataFrame(all_metrics)
display(df_all_metrics)

In [ ]:
import numpy as np

# Let's take one original vocal track
song_name = list(processed_results.keys())[0]
song_folder = os.path.join(musdb_path, song_name)
stems_baseline, _ = load_custom_stems(song_folder)

if 'vocals' in stems_baseline:
    original = stems_baseline['vocals'][0]

    # Comparing the signal to itself (the 'original' state)
    baseline_psnr = calculate_psnr(original, original)
    baseline_snr = calculate_snr(original, original)

    print(f"Baseline Metrics (Original vs Original) for {song_name}:")
    print(f" - PSNR: {baseline_psnr}")
    print(f" - SNR Proxy: {baseline_snr}")

### Testing with a Specific Folder
Use this section to process a specific subset or a new folder containing your 4 songs.

In [ ]:
# Update this path to your folder with 4 songs
test_folder_path = '/content/drive/MyDrive/Minor Testing'

if os.path.exists(test_folder_path):
    test_songs = [f for f in os.listdir(test_folder_path) if os.path.isdir(os.path.join(test_folder_path, f))]
    print(f"Found {len(test_songs)} song folders for testing.")

    test_results = {}
    for song in test_songs:
        song_p = os.path.join(test_folder_path, song)
        stems_t, sr_t = load_custom_stems(song_p)

        if 'vocals' in stems_t:
            print(f"Enhancing: {song}...")
            enhanced_vocal = apply_multiband_compression(stems_t['vocals'][0], sr_t)
            test_results[song] = enhanced_vocal

    print(f"\nProcessed {len(test_results)} songs from test folder.")
else:
    print("Path not found. Please update 'test_folder_path' with the correct Drive path.")

### Recommended Test Suite Setup
Defining a specific list of songs to use as a gold-standard test set for evaluating the hearing enhancement profiles.

In [ ]:
recommended_test_songs = [
    'The Districts - Vermont',
    'AvaLuna - Waterduct',
    'Hezekiah Jones - Borrowed Heart',
    'Alexander Ross - Goodbye Bolero',
    'Actions - South Of The Water'
]

print(f"Selected {len(recommended_test_songs)} tracks for testing.")

# Filter the processed results or load them specifically for evaluation
test_evaluation_data = {name: processed_results[name] for name in recommended_test_songs if name in processed_results}
print(f"Ready to evaluate: {list(test_evaluation_data.keys())}")

In [ ]:
# Let's list the top-level folders in your Drive to help find the correct path
import os
drive_root = '/content/drive/MyDrive'
if os.path.exists(drive_root):
    print("Folders found in your Google Drive:")
    for item in os.listdir(drive_root):
        if os.path.isdir(os.path.join(drive_root, item)):
            print(f" - {os.path.join(drive_root, item)}")
else:
    print("Drive not mounted. Please run the Drive mount cell first.")

In [ ]:
import os

drive_root = '/content/drive/MyDrive'
training_folder_name = 'Minor Training'
training_path = os.path.join(drive_root, training_folder_name)

if os.path.exists(drive_root):
    print(f"Searching for '{training_folder_name}' in {drive_root}...")
    contents = os.listdir(drive_root)
    if training_folder_name in contents:
        songs = [f for f in os.listdir(training_path) if os.path.isdir(os.path.join(training_path, f)) and f != 'Enhanced_Outputs']
        print(f'Successfully located training folder. Current songs ({len(songs)}):')
        for s in songs:
            print(f' - {s}')
    else:
        print(f"Folder '{training_folder_name}' not found. Available folders are:")
        for item in contents:
            if os.path.isdir(os.path.join(drive_root, item)):
                print(f" - {item}")
else:
    print('Google Drive root not accessible. Please ensure Drive is mounted.')

In [ ]:
import os

drive_root = '/content/drive/MyDrive'
training_path = os.path.join(drive_root, 'Minor Training')

if os.path.exists(training_path):
    # Re-scanning the directory to pick up the 101 new tracks
    all_items = os.listdir(training_path)
    songs = [f for f in all_items if os.path.isdir(os.path.join(training_path, f)) and f != 'Enhanced_Outputs']

    print(f"Successfully verified dataset.")
    print(f"Total training songs found: {len(songs)}")

    if len(songs) > 0:
        print("First 5 tracks detected:")
        for s in songs[:5]:
            print(f" - {s}")
else:
    print('Training folder not found. Please ensure Google Drive is mounted and the folder name is correct.')

In [ ]:
import os

def count_all_song_folders(path):
    song_dirs = []
    for root, dirs, files in os.walk(path):
        # Check if this directory contains 'vocals.wav' or 'mixture.wav' - common markers
        if 'vocals.wav' in files or 'mixture.wav' in files:
            song_dirs.append(root)
    return song_dirs

all_detected_songs = count_all_song_folders(training_path)
print(f"Total unique song folders found recursively: {len(all_detected_songs)}")
if len(all_detected_songs) > 0:
    print("Sample of paths found:")
    for p in all_detected_songs[:5]:
        print(f" - {p}")

In [ ]:
import os

def scan_for_any_audio(path):
    audio_extensions = ('.wav', '.mp3', '.flac', '.m4a')
    found_files = []
    for root, dirs, files in os.walk(path):
        for file in files:
            if file.lower().endswith(audio_extensions):
                found_files.append(os.path.join(root, file))
    return found_files

all_audio_files = scan_for_any_audio(training_path)
print(f"Total audio files found: {len(all_audio_files)}")

# Group by folder to see distribution
folders_with_audio = sorted(list(set([os.path.dirname(f) for f in all_audio_files])))
print(f"Total folders containing audio: {len(folders_with_audio)}")

if len(folders_with_audio) > 80:
    print("New folders detected that weren't in the previous count:")
    # Display some folders that might be the 'missing' ones
    for f in folders_with_audio[80:90]:
        print(f" - {f}")

In [ ]:
import os

# Get a full list of every single directory inside Minor Training
all_subdirs = []
for root, dirs, files in os.walk(training_path):
    all_subdirs.append(root)

print(f"Total subdirectories scanned: {len(all_subdirs)}")
print("\nFirst 20 directories found:")
for d in all_subdirs[:20]:
    print(f" - {d}")

# Specifically look for folders that were not in our previous 'folders_with_audio' list
folders_with_audio_set = set(folders_with_audio)
empty_or_new_dirs = [d for d in all_subdirs if d not in folders_with_audio_set]

print(f"\nFound {len(empty_or_new_dirs)} directories that don't directly contain supported audio files.")

In [ ]:
import os

# Scanning for .stem.mp4 files specifically
stem_files = []
for root, dirs, files in os.walk(training_path):
    for file in files:
        if file.endswith('.stem.mp4'):
            stem_files.append(os.path.join(root, file))

print(f"Total .stem.mp4 files found: {len(stem_files)}")
if stem_files:
    print("\nFirst 10 stem files detected:")
    for f in stem_files[:10]:
        print(f" - {os.path.basename(f)}")
else:
    print("\nNo .stem.mp4 files found. The files might still be syncing or in a different format.")

In [ ]:
import musdb
import matplotlib.pyplot as plt
import numpy as np

# Note: To use the full dataset, you'd provide the root path:
# mus = musdb.DB(root="/path/to/musdb18")

print("musdb library imported successfully.")

### Download Sample Data
To begin experimenting, we can download the small preview version of the dataset (MUSDB18-7s).

In [ ]:
import musdb

# This downloads the small 7-second preview set to the current directory
mus = musdb.DB(download=True)

# List the first few tracks available to verify
print(f"Number of tracks loaded: {len(mus)}")
for track in mus.tracks[:3]:
    print(f"Track: {track.name} | Stems: {list(track.targets.keys())}")

### Download Sample Data
If you are just starting, we can download the preview version of the dataset.

In [ ]:
import musdb

# 'substems' is not a valid argument for DB().
# The download has already completed in the previous cell.
mus = musdb.DB(download=True)

# List the first few tracks available
print(f"Number of tracks loaded: {len(mus)}")
for track in mus.tracks[:3]:
    print(f"Track: {track.name} | Stems: {list(track.targets.keys())}")

### Visualize and Process Audio
We will now load a specific track and visualize its stems using a spectrogram.

In [ ]:
import scipy.signal

# Select the first track
track = mus.tracks[0]

# Get the 'vocals' stem audio data
vocals_audio = track.targets['vocals'].audio
fs = track.rate

# Compute spectrogram
f, t, Sxx = scipy.signal.spectrogram(vocals_audio[:, 0], fs)

plt.figure(figsize=(10, 4))
plt.pcolormesh(t, f, 10 * np.log10(Sxx), shading='gouraud')
plt.ylabel('Frequency [Hz]')
plt.xlabel('Time [sec]')
plt.title(f'Spectrogram of Vocals: {track.name}')
plt.colorbar(label='Intensity [dB]')
plt.ylim(0, 8000)  # Human voice range
plt.show()

### Basic Frequency Enhancement
We will apply a high-shelf filter to boost frequencies where hearing loss is most common (typically > 3000 Hz).

In [ ]:
from scipy.signal import butter, lfilter

def high_shelf_filter(data, cutoff, fs, gain_db):
    # Design a basic filter to boost high frequencies
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    # Using a simple Butterworth highpass as a proxy for high-shelf
    b, a = butter(1, normal_cutoff, btype='highpass', analog=False)
    boosted = lfilter(b, a, data)

    # Combine original and filtered (boosted) signal
    gain_linear = 10**(gain_db / 20)
    return data + (boosted * (gain_linear - 1))

# Apply 10dB boost above 3000Hz
boosted_vocals = high_shelf_filter(vocals_audio[:, 0], 3000, fs, 10)

plt.figure(figsize=(10, 4))
plt.plot(vocals_audio[:2000, 0], label='Original', alpha=0.7)
plt.plot(boosted_vocals[:2000], label='Boosted (>3kHz)', alpha=0.7)
plt.title("Waveform Comparison: Original vs Boosted Vocals (First 2000 samples)")
plt.legend()
plt.show()

In [ ]:
import os
import librosa
import numpy as np
from scipy.signal import butter, lfilter

def load_custom_stems(song_path):
    """Loads vocal stems from a specific folder."""
    stems = {}
    sr = 44100
    found_any = False
    for stem_name in ['vocals', 'drums', 'bass', 'other', 'mixture']:
        file_path = os.path.join(song_path, f"{stem_name}.wav")
        if os.path.exists(file_path):
            audio, loaded_sr = librosa.load(file_path, sr=None, mono=False)
            # Ensure stereo for consistency if mono was loaded
            if audio.ndim == 1:
                audio = np.array([audio, audio])
            stems[stem_name] = audio
            sr = loaded_sr
            found_any = True
    return stems, sr

def high_shelf_filter(data, cutoff, fs, gain_db):
    """Applies a high-frequency boost."""
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(1, normal_cutoff, btype='highpass', analog=False)
    boosted = lfilter(b, a, data)
    gain_linear = 10**(gain_db / 20)
    return data + (boosted * (gain_linear - 1))

def improvise_enhancement(audio, sr, profile='mild'):
    """Applies profile-based multi-band enhancement."""
    nyq = 0.5 * sr
    if profile == 'mild':
        crossover, gain = 2000, 1.8
    elif profile == 'moderate':
        crossover, gain = 1500, 2.5
    else: # severe
        crossover, gain = 1000, 4.0

    b_low, a_low = butter(1, crossover/nyq, btype='lowpass')
    b_high, a_high = butter(1, crossover/nyq, btype='highpass')
    low = lfilter(b_low, a_low, audio)
    high = lfilter(b_high, a_high, audio)
    return low + (high * gain)

def combine_stems(stems_dict, stem_names):
    """Combines specified audio stems by summing their audio data."""
    combined_audio = None
    for name in stem_names:
        if name in stems_dict:
            audio = stems_dict[name]
            if combined_audio is None:
                combined_audio = np.zeros_like(audio)
            # Ensure shapes match before summing (e.g., stereo vs stereo)
            if combined_audio.shape == audio.shape:
                combined_audio += audio
            else:
                # Handle potential mono/stereo mismatch by duplicating mono to stereo
                if audio.ndim == 1 and combined_audio.ndim == 2 and combined_audio.shape[0] == 2:
                    audio_stereo = np.array([audio, audio])
                    if audio_stereo.shape == combined_audio.shape:
                        combined_audio += audio_stereo
                    else:
                        print(f"Warning: Skipping {name} due to shape mismatch after mono conversion.")
                elif audio.ndim == 2 and combined_audio.ndim == 1 and audio.shape[0] == 2:
                    # Convert combined_audio to stereo if current audio is stereo
                    combined_audio = np.array([combined_audio, combined_audio])
                    if audio.shape == combined_audio.shape:
                        combined_audio += audio
                    else:
                        print(f"Warning: Skipping {name} due to shape mismatch after combined_audio conversion.")
                else:
                    print(f"Warning: Skipping {name} due to shape mismatch: {audio.shape} vs {combined_audio.shape}")
        else:
            print(f"Warning: Stem '{name}' not found in the provided stems dictionary.")

    if combined_audio is not None:
        # Normalize to prevent clipping, assuming it's float audio
        max_abs_val = np.max(np.abs(combined_audio))
        if max_abs_val > 0:
            combined_audio = combined_audio / max_abs_val

    return combined_audio


print("Project functions initialized and ready for use.")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Audio
import numpy as np
import os

# Configuration
musdb_path = '/content/drive/MyDrive/Minor Training'

# Scan for songs
try:
    if os.path.exists(musdb_path):
        song_list = [f for f in os.listdir(musdb_path) if os.path.isdir(os.path.join(musdb_path, f)) and f != 'Enhanced_Outputs']
    else:
        print("Path not found. Please ensure Google Drive is mounted.")
        song_list = []
except Exception as e:
    print(f"Error accessing path: {e}")
    song_list = []

# UI Components
song_dropdown = widgets.Dropdown(options=song_list, description='Select Song:')
profile_toggle = widgets.ToggleButtons(options=['mild', 'moderate', 'severe'], description='Profile:')
process_button = widgets.Button(description='Enhance & Play', button_style='success')
output_area = widgets.Output()

def on_button_clicked(b):
    with output_area:
        output_area.clear_output()
        song_name = song_dropdown.value
        profile = profile_toggle.value

        if not song_name:
            print("Please select a song first.")
            return

        print(f"Processing '{song_name}' with '{profile}' profile...")

        # Load the song
        path = os.path.join(musdb_path, song_name)
        stems_ui, sr_ui = load_custom_stems(path)

        if 'vocals' in stems_ui:
            vocal = stems_ui['vocals'][0] # Taking first channel for processing
            # Apply enhancement
            enhanced = improvise_enhancement(vocal, sr_ui, profile=profile)

            # Normalize for playback
            enhanced_norm = enhanced / (np.max(np.abs(enhanced)) + 1e-9)

            print("--- Original Vocal ---")
            display(Audio(vocal, rate=sr_ui))

            print(f"--- Enhanced Vocal ({profile} profile) ---")
            display(Audio(enhanced_norm, rate=sr_ui))

            # Demonstrate combining stems
            stems_to_combine = ['vocals', 'drums', 'bass', 'other', 'mixture']
            combined_audio = combine_stems(stems_ui, stems_to_combine)
            if combined_audio is not None:
                print(f"--- Combined Audio ({', '.join(stems_to_combine)}) ---")
                display(Audio(combined_audio, rate=sr_ui))
            else:
                print("Could not combine stems.")

        else:
            print("Could not find vocal stems for this song.")

process_button.on_click(on_button_clicked)

ui_layout = widgets.VBox([song_dropdown, profile_toggle, process_button, output_area])
display(ui_layout)

In [ ]:
import os
import numpy as np
from IPython.display import Audio, display

# Define the target song from your Minor Training folder
song_name = 'The Districts - Vermont'
song_path = os.path.join(musdb_path, song_name)

print(f"Processing all stems for: {song_name}")

# 1. Load all available stems
stems_all, sr_all = load_custom_stems(song_path)

# 2. Combine specific stems: vocals, drums, bass, and other
# We exclude 'mixture' from the sum because 'mixture' is usually the pre-mixed sum of the others
stems_to_sum = ['vocals', 'drums', 'bass', 'other']
combined_audio = combine_stems(stems_all, stems_to_sum)

if combined_audio is not None:
    print(f"Successfully combined {list(stems_all.keys())} stems.")

    # Display playback for the combined result
    display(Audio(combined_audio, rate=sr_all))

    # Optional: Compare with the original 'mixture.wav' file provided in the dataset
    if 'mixture' in stems_all:
        print("--- Original Mixture for Comparison ---")
        display(Audio(stems_all['mixture'], rate=sr_all))
else:
    print("Failed to combine stems. Please check if the .wav files exist in the folder.")

In [ ]:
import numpy as np
from IPython.display import Audio, display

# 1. Load stems for 'The Districts - Vermont'
song_path = os.path.join(musdb_path, 'The Districts - Vermont')
stems, sr = load_custom_stems(song_path)

if 'vocals' in stems:
    # 2. Enhance the vocals specifically (using 'severe' profile as an example)
    # We enhance only the first channel for simplicity in this demonstration
    vocal_channel = stems['vocals'][0]
    enhanced_vocal = improvise_enhancement(vocal_channel, sr, profile='severe')

    # Convert enhanced vocal to stereo to match other stems
    enhanced_vocal_stereo = np.array([enhanced_vocal, enhanced_vocal])

    # 3. Create a copy of the stems dict and replace the original vocal with the enhanced one
    enhanced_stems_dict = stems.copy()
    enhanced_stems_dict['vocals'] = enhanced_vocal_stereo

    # 4. Combine ALL specified stems
    stems_to_combine = ['vocals', 'drums', 'bass', 'other', 'mixture']
    final_enhanced_mix = combine_stems(enhanced_stems_dict, stems_to_combine)

    print("Successfully created enhanced mix including enhanced vocals + all other stems.")
    display(Audio(final_enhanced_mix, rate=sr))
else:
    print("Vocal stem not found to apply enhancement.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

output_folder = '/content/drive/MyDrive/Enhanced_Outputs'
os.makedirs(output_folder, exist_ok=True)

def enhance_and_save(audio_path, profile):
    y, sr = librosa.load(audio_path, sr=None)
    # ... your enhancement logic ...
    out_path = f"{output_folder}/enhanced_{profile}_{os.path.basename(audio_path)}"
    sf.write(out_path, y, sr)
    return out_path  # gradio plays this back and Drive stores it

In [ ]:
import subprocess

result = subprocess.run(
    ["find", "/content/drive/MyDrive", "-name", "*.ipynb"],
    capture_output=True, text=True
)
print("Found notebooks:\n", result.stdout)

In [ ]:
!git reset --hard HEAD~1

In [ ]:
!grep -r "ghp_" .

In [ ]:
!git add .
!git commit -m "Clean version without secrets"

In [ ]:
!git remote set-url origin https://sHiV-0420:@github.com/Shiv-0420/music-improvisation-hearing-loss.git

In [ ]:
!git push origin main --force

In [ ]:
!git remote set-url origin https://github.com/Shiv-0420/music-improvisation-hearing-loss.git

In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/Music Improvisation for Hearing Loss.ipynb" \
"/content/music-improvisation-hearing-loss/"

In [ ]:
!ls

In [ ]:
!pip install nbstripout
!nbstripout Music_Improvisation_for_Hearing_Loss.ipynb

In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks"